<a href="https://colab.research.google.com/github/Mehroz485/ML-01/blob/main/work/notebooks/w01_research_question_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mehroz485/ML-01/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Setup: locate the repo whether we're running inside the cloned repo (normal Jupyter)
# or from a standalone .ipynb with no repo on disk (Google Colab). Every later cell
# reads files through REPO_ROOT so the notebook runs in either place unchanged.
import os
import subprocess

def _find_repo_root():
    cur = os.getcwd()
    for _ in range(5):
        if os.path.exists(os.path.join(cur, "data", "raw", "content_refresh_anonymized.csv")):
            return cur
        cur = os.path.dirname(cur)
    if not os.path.exists("ML-01"):
        subprocess.run(
            ["git", "clone", "https://github.com/Mehroz485/ML-01.git"], check=True
        )
    return os.path.abspath("ML-01")

REPO_ROOT = _find_repo_root()
print("REPO_ROOT:", REPO_ROOT)


REPO_ROOT: /content/ML-01


## 1. My lane (or freestyle) and why

**Lane 2 — Refresh / Content Opportunity Scoring.**

I already ran the full starter pipeline (`scripts/01`–`05`) end to end, so this lane's shape —
baseline score → model → ranked queue with reason codes — is the one I understand best and can
push further fastest. It also matches how I like to build things: a system that hands a person a
short, ordered, explainable list to act on, not just a report saying "these signals correlate."

**Why not just keep the starter's rule-based baseline (why ML earns its place here):** the starter
pipeline already has six hand-written reason-code rules (`stale_visible_page`,
`declining_with_demand`, `thin_visible_page`, `page_one_decay_risk`, `low_ctr_visible_page`,
`low_engagement_visible_page`). The problem is these rules overlap heavily and, on their own, flag
most of the dataset (see the numbers below) — a rule that says "look at 70% of your pages" is not
a decision aid, it's a bigger pile. The signals (trend, CTR, position, engagement, freshness) are
real but tangled: a page can be declining *and* still have a decent CTR, or thin *and* still rank
well. That's exactly the "real pattern, too messy to hand-rank" case the framing skill flags as
ML's actual job — and it's not hypothetical here: the committed pipeline output
(`outputs/model_report.md`) already shows a random forest beating the rule baseline on precision@50
on this exact data (confirmed again below, not just quoted from memory).

I'm keeping **Lane 1 (Ranking Signal Analysis)** as my fallback: if the ML-05 leakage audit shows
`is_declining_label` is too weak a proxy to keep pushing on, dropping the ranking/queue goal and
reporting signal relationships instead is a small pivot, not a restart, since Lane 2's signal audit
(ML-06) already produces most of what Lane 1 needs.


In [2]:
# Section 1 support: does the data actually back "32 real clients, usable volume" or not?
import pandas as pd

df = pd.read_csv(os.path.join(REPO_ROOT, "data/raw/content_refresh_anonymized.csv"))

print("rows, cols:", df.shape)
print("clients:", df["client_id"].nunique())
print("median rows per client:", df.groupby("client_id").size().median())
print("declining rate (is_declining_label proxy):",
      round((df["trend_direction"] == "down").mean(), 3))


rows, cols: (30000, 44)
clients: 32
median rows per client: 567.0
declining rate (is_declining_label proxy): 0.542


## 2. The question: decision, action, cost of a wrong call

**Research question:** given a fixed weekly review capacity, which content items should FlyRank's
content team look at first — and with what suggested action?

**Unit of analysis:** one row = one pseudonymized content item (a "page"), scoped to one client,
described by its trailing-90-day window. Not a client, not a day — one page, one snapshot. (The
warehouse release lets me rebuild this same grain with real time windows instead of one fixed
90-day snapshot, if I move to it after Week 4.)

**Output:** a ranked queue — page id, score, suggested action (`refresh`,
`refresh_and_review_ctr`, `refresh_and_review_engagement`, `expand_and_refresh`, `monitor`), and the
reason codes that explain *why* it's on the list.

**Who acts, and what they do:** a content strategist / reviewer on FlyRank's team opens the queue
and works top-down until their capacity for the week runs out. The reason codes let them start
from "why is this here" instead of diagnosing a page cold.

**Cost of a wrong call, both directions:**
- *False positive* — a page is pushed to the top of the queue but nothing was really wrong. The
  reviewer spends real hours (reading, maybe drafting a rewrite, maybe looping in a writer) on a
  page that would have been fine left alone. That hour didn't go to a page that needed it.
- *False negative* — a page that's genuinely declining, with real demand behind it, gets buried
  or missed. The client keeps losing impressions/clicks/sessions on a page worth protecting, for
  as long as it takes someone to notice another way.
- Because review capacity is small next to the candidate pool (most of the dataset trips at least
  one "worth a look" rule — numbers below), the hard part isn't *finding* declining pages, a plain
  rule already does that. It's **ordering**: which handful get looked at *this* week. That's a
  ranking problem, which is why precision@K — not raw accuracy — is the metric that matches the
  actual decision.


In [3]:
# Section 2 support: how big is the "worth a look" pool vs. a realistic weekly review capacity?
flagged = (
    ((df["trend_direction"] == "down") & (df["impressions_90d"] >= 100))              # declining_with_demand
    | ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500))         # stale_visible_page
    | ((df["word_count"] > 0) & (df["word_count"] < 1200) & (df["impressions_90d"] >= 250))  # thin_visible_page
    | ((df["avg_position"] > 0) & (df["avg_position"] <= 10) & (df["content_age_days"] >= 180))  # page_one_decay_risk
    | ((df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["ctr"] < 0.5))  # low_ctr_visible_page
    | ((df["sessions_90d"] >= 30) & ((df["engagement_rate"] < 30) | (df["scroll_rate"] < 30)))  # low_engagement_visible_page
)

n_flagged = int(flagged.sum())
weekly_capacity = 50  # a reviewer working top-50 candidates a week, matches the lane guide's precision@50 framing

print(f"rows tripping >= 1 starter rule: {n_flagged} / {len(df)} ({n_flagged / len(df):.1%})")
print(f"at a capacity of {weekly_capacity}/week, the flagged pool alone is a "
      f"{n_flagged / weekly_capacity:.0f}-week backlog if every flagged page were reviewed once")
print("median impressions_90d among flagged rows:", int(df.loc[flagged, "impressions_90d"].median()))


rows tripping >= 1 starter rule: 21024 / 30000 (70.1%)
at a capacity of 50/week, the flagged pool alone is a 420-week backlog if every flagged page were reviewed once
median impressions_90d among flagged rows: 1729


## 3. Quick look at the data (2-3 real numbers)

Three numbers from the 30,000-row starter dataset back this lane (computed in the cell below, not
typed from memory):

1. **32 clients, a median of 567 rows each** — real client diversity, not one account's pattern
   dressed up as general.
2. **13,152 rows (43.8%) are `declining_with_demand`** (`trend_direction == "down"` and
   `impressions_90d >= 100`) — a large, real pool of candidates with actual search demand behind
   them, not noise.
3. **21,024 rows (70.1%) trip at least one starter reason-code rule**, yet the committed pipeline
   run (`outputs/model_report.md`) shows a random forest hits **precision@50 = 0.740** against the
   rule baseline's **0.240** — roughly 37 of the top 50 right vs. 12 of 50. That gap, on this exact
   data, is the concrete evidence that ranking beats a flat rule here, not just a theory.


In [4]:
# Section 3: the "why this lane is worth 7 weeks" numbers, pulled from real files, not memory.
import re

declining_with_demand = ((df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)).sum()
print(f"declining_with_demand: {declining_with_demand} rows "
      f"({declining_with_demand / len(df):.1%} of {len(df)})")
print(f"any starter reason-code flag: {n_flagged} rows ({n_flagged / len(df):.1%})")

# Pull the already-verified model comparison straight out of the committed report,
# so the precision@50 numbers below are read from the file, not recalled from the guide.
report = open(os.path.join(REPO_ROOT, "outputs/model_report.md")).read()
rows = re.findall(r"\|\s*(baseline_rules|random_forest)\s*\|([^\n]+)", report)
for name, rest in rows:
    cols = [c.strip() for c in rest.strip().strip("|").split("|")]
    print(f"{name}: ROC AUC={cols[0]}, avg precision={cols[1]}, precision@50={cols[2]}")


declining_with_demand: 13152 rows (43.8% of 30000)
any starter reason-code flag: 21024 rows (70.1%)
random_forest: ROC AUC=0.750, avg precision=0.618, precision@50=0.740
baseline_rules: ROC AUC=0.627, avg precision=0.468, precision@50=0.240


## 4. Careful words: what I can and can't claim

**What this will be able to say:** a **decision-support** ranking, built only from *observed*
signals (impressions, clicks, CTR, position, sessions, engagement/scroll rate, age, freshness) —
never from FlyRank's own precomputed decision flags, since `health_score`, `priority_score`, and
`action_type` aren't even shipped in this dataset. Any precision@K, recall, or AUC I report
describes how well the ranking recovers pages that already matched a defined trend/CTR/engagement
pattern in a trailing window, on a client-holdout split — an internal, retrospective check, not a
live-traffic result.

**What this will not say:**
- That the model predicts Google's algorithm, ranking factors, or AI-search visibility — nothing
  here touches how Google or an AI engine actually ranks anything.
- That refreshing a page *causes* recovery — there's no experiment in this data (no A/B test, no
  matched refreshed-vs-untouched comparison), so any recovery seen after a refresh stays
  correlational at best.
- That `is_declining_label` (`trend_direction == "down"`) is the true target. The lane guide is
  explicit that it's a proxy computed from the *current* window, not a future outcome. If I'm still
  using it past Week 4, I'll say so plainly as "reproducing the starter's rule," and I'm treating a
  future-window redefinition (prior 90 days → next 30 days decline) as the honest upgrade to move
  toward once I reach the warehouse release.
- That a wrong call has a dollar cost — I only have review-hours and traffic counts, not revenue,
  so cost claims stay in "hours" and "impressions/sessions," not "$."


In [5]:
# Section 4 support: the label-trap check — confirm the columns the label is BUILT FROM
# are excluded from any feature list, since a leaky feature would make the "decision-support,
# not causal" framing above meaningless (the model would just be reading its own answer).
label_source_cols = {"trend_direction", "trend_pct"}
candidate_features = set(df.columns) - label_source_cols - {"content_id", "client_id"}

print("label source columns (never features):", label_source_cols)
print("is_declining_label true rate:", round((df['trend_direction'] == 'down').mean(), 3))
print("usable feature columns available:", len(candidate_features))


label source columns (never features): {'trend_direction', 'trend_pct'}
is_declining_label true rate: 0.542
usable feature columns available: 40


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
